In [ ]:
import pyspark
from pyspark.sql import SparkSession

In [ ]:
import os
# Tell Jupyter to use Git Bash's shell engine
os.environ['SHELL'] = r'C:\Program Files\Git\bin\sh.exe'

# Set HADOOP_HOME to the parent folder of 'bin'
os.environ['HADOOP_HOME'] = "C:/hadoop"
# Add the bin folder to the system PATH
os.environ['PATH'] += os.pathsep + "C:/hadoop/bin"

import sys

# Get the path to the current python executable in your .venv
python_path = sys.executable

os.environ['PYSPARK_PYTHON'] = python_path
os.environ['PYSPARK_DRIVER_PYTHON'] = python_path

In [ ]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .config("spark.hadoop.fs.permissions.umask-mode", "000") \
    .getOrCreate()

In [ ]:
# import wget
# url = 'https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz'
# filename = wget.download(url)

In [ ]:
import gzip

# Decompress the gzipped file and write to a new CSV file
with gzip.open('fhvhv_tripdata_2021-01.csv.gz', 'rb') as f_in:
    with open('fhvhv_tripdata_2021-01.csv', 'wb') as f_out:
        f_out.write(f_in.read())

In [ ]:
df = spark.read \
    .option("header", "true") \
    .csv('fhvhv_tripdata_2021-01.csv')

In [ ]:
df.schema

In [ ]:
with open('fhvhv_tripdata_2021-01.csv', 'r') as f_in, open('head.csv', 'w') as f_out:
    for i in range(1001):
        line = f_in.readline()
        f_out.write(line)

In [ ]:
import pandas as pd

In [ ]:
df_pandas = pd.read_csv('head.csv')

In [ ]:
df_pandas.dtypes

In [ ]:
spark.createDataFrame(df_pandas).schema

Integer - 4 bytes
Long - 8 bytes

In [ ]:
from pyspark.sql import types

In [ ]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [ ]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv')

In [ ]:
df = df.repartition(24)

In [ ]:
df.write.mode("overwrite").parquet('fhvhv/2021/01/')

In [ ]:
df = spark.read.parquet('fhvhv/2021/01/')

In [ ]:
df.printSchema()

SELECT * FROM df WHERE hvfhs_license_num =  HV0003

In [ ]:
from pyspark.sql import functions as F

In [ ]:
df.show()

In [ ]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
  .filter(df.hvfhs_license_num == 'HV0005')

In [ ]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [ ]:
crazy_stuff('B02884')

In [ ]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [ ]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', 
        F.when(F.substring('dispatching_base_num', 2, 5).cast('int') % 7 == 0, 
            F.concat(F.lit("s/ "), F.lpad(F.substring('dispatching_base_num', 2, 5), 3, '0')))
        .when(F.substring('dispatching_base_num', 2, 5).cast('int') % 3 == 0, 
            F.concat(F.lit("a/ "), F.lpad(F.substring('dispatching_base_num', 2, 5), 3, '0')))
        .otherwise(F.concat(F.lit("e/ "), F.lpad(F.substring('dispatching_base_num', 2, 5), 3, '0')))) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()